# GNN-BERT Music Context — Demo Notebook (Task 2)

Loads the trained Task 2 GraphSAGE model plus one saved segment graph,
and runs a single end-to-end inference: predicting genre purely from audio structure (no text).

In [ ]:
import sys
sys.path.append('/content/gnn-bert-music-context/src')

import torch
import pandas as pd
from torch_geometric.data import Batch as PyGBatch

from gnn_model import GraphSAGEClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Recover genre label ordering (matches training-time LabelEncoder order)
METADATA_DIR = f"{PROJECT_DIR}/data/raw/fma_metadata"
tracks = pd.read_csv(f"{METADATA_DIR}/tracks.csv", index_col=0, header=[0, 1])
small = tracks[tracks[('set', 'subset')] == 'small']
GENRE_CLASSES = sorted(small[('track', 'genre_top')].dropna().unique())
print(GENRE_CLASSES)

In [ ]:
# Load one example graph from the saved graph set
all_graphs = torch.load(f"{PROJECT_DIR}/data/graphs/all_graphs.pt", weights_only=False)
graph = all_graphs[0]
print(graph)
print("True label index:", graph.y.item(), "->", GENRE_CLASSES[graph.y.item()])

In [ ]:
# Load the trained GraphSAGE model (Task 2 checkpoint)
model = GraphSAGEClassifier(num_classes=len(GENRE_CLASSES)).to(device)
model.load_state_dict(torch.load(f"{PROJECT_DIR}/results/task2_gnn_model.pt", map_location=device))
model.eval()
print("Model loaded.")

In [ ]:
# Run one end-to-end inference: graph structure -> predicted genre
graph_batch = PyGBatch.from_data_list([graph]).to(device)

with torch.no_grad():
    logits = model(graph_batch.x, graph_batch.edge_index, graph_batch.batch)
    pred_idx = torch.argmax(logits, dim=1).item()

print(f"True genre:      {GENRE_CLASSES[graph.y.item()]}")
print(f"Predicted genre: {GENRE_CLASSES[pred_idx]}")

## Note

This demo uses Task 2's GraphSAGE model, which classifies genre from chord/segment
graph structure alone (no text). Raw FMA metadata is used only to recover genre label
ordering — the model itself only needs the graph and its saved weights.